# Tarea B — Temperatura, top-p y entropía de la distribución de salida

MMIA 6013 · Tarea de práctica para el solver del Taller 03 v2

Un modelo de lenguaje produce, para el siguiente token, un vector de logits. Consideramos un vocabulario de seis tokens, t0 a t5, con estos logits:

| Token | t0 | t1 | t2 | t3 | t4 | t5 |
|---|---|---|---|---|---|---|
| Logit | 2.0 | 1.0 | 0.5 | 0.2 | -1.0 | -3.0 |

El notebook se organiza en cuatro partes: (1) softmax con temperatura y entropía en bits, (2) muestreo de núcleo (top-p), (3) comprobación empírica por muestreo y divergencia KL, y (4) una pregunta conceptual sobre los límites de la temperatura. Todas las cifras se calculan en las celdas de código y quedan visibles en sus salidas.

## Parte 1 — Softmax con temperatura

Implementamos en NumPy la función softmax con temperatura, p_i = exp(z_i/T) / Σ_j exp(z_j/T), de forma numéricamente estable: antes de exponenciar se resta el máximo de los logits (restar el máximo antes de dividir entre T es una formulación estable equivalente), lo que evita desbordamientos sin alterar el resultado. Calculamos la distribución para T = 0.5, T = 1 y T = 2, y la entropía de Shannon en bits (logaritmo base 2) de cada una. La tabla de la salida presenta las tres distribuciones y las tres entropías con cuatro decimales, junto con la verificación de que cada fila sume 1.

In [1]:
import json
import numpy as np

# ------------------------------------------------------------------
# Datos de la tarea (Parte 1): vocabulario de seis tokens y logits
# ------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
z = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)


# ------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable:
# p_i = exp((z_i - max(z))/T) / sum_j exp((z_j - max(z))/T)
# (restar el máximo antes de dividir por T evita overflow/underflow)
# ------------------------------------------------------------------
def softmax_temperature(logits, T):
    logits = np.asarray(logits, dtype=float)
    T = float(T)
    if T <= 0:
        raise ValueError("La temperatura debe ser positiva.")
    z_shift = (logits - np.max(logits)) / T  # estabilidad numérica
    e = np.exp(z_shift)
    return e / np.sum(e)


# ------------------------------------------------------------------
# Entropía de Shannon en bits: H = -sum p_i * log2(p_i)
# ------------------------------------------------------------------
def entropy_bits(p):
    p = np.asarray(p, dtype=float)
    nz = p[p > 0]  # los términos con p=0 no contribuyen
    return float(-np.sum(nz * np.log2(nz)))


# ------------------------------------------------------------------
# Cálculo de las distribuciones y entropías para T = 0.5, 1, 2
# ------------------------------------------------------------------
temperatures = [0.5, 1.0, 2.0]

distributions = {}
entropies = {}
row_sums = {}
for T in temperatures:
    p = softmax_temperature(z, T)
    distributions[T] = p
    entropies[T] = entropy_bits(p)
    row_sums[T] = float(np.sum(p))  # verificación: debe ser 1.0

# ------------------------------------------------------------------
# Tabla única con las tres distribuciones y las tres entropías
# (pandas no está disponible: se formatea manualmente a 4 decimales)
# ------------------------------------------------------------------
header = ["T"] + tokens + ["entropia_bits"]
rows = []
for T in temperatures:
    row = [f"{T:g}"] + [f"{v:.4f}" for v in distributions[T]] + [f"{entropies[T]:.4f}"]
    rows.append(row)

widths = [max(len(header[c]), max(len(r[c]) for r in rows)) for c in range(len(header))]


def fmt_row(r):
    return " | ".join(r[c].rjust(widths[c]) for c in range(len(header)))


sep = "-+-".join("-" * w for w in widths)
table_str = "\n".join([fmt_row(header), sep] + [fmt_row(r) for r in rows])

print("Parte 1 - Softmax con temperatura: distribuciones y entropia (bits)")
print(table_str)

print()
print("Verificacion (suma de cada fila):")
for T in temperatures:
    print(f"  T={T:g}: suma = {row_sums[T]:.4f}")

print()
print("Valores (mismos numeros que en la tabla):")
for T in temperatures:
    dist_str = ", ".join(f"{tok}={distributions[T][i]:.4f}" for i, tok in enumerate(tokens))
    print(f"  T={T:g}: {dist_str} | entropia={entropies[T]:.4f} bits")

# ------------------------------------------------------------------
# Resultados a resultados.json (solo floats/strings de Python)
# ------------------------------------------------------------------
results = {
    "logits": {tok: float(val) for tok, val in zip(tokens, z)},
    "temperatures": [float(T) for T in temperatures],
    "distributions": {
        f"T={T:g}": {tok: float(distributions[T][i]) for i, tok in enumerate(tokens)}
        for T in temperatures
    },
    "entropies_bits": {f"T={T:g}": float(entropies[T]) for T in temperatures},
    "row_sums": {f"T={T:g}": float(row_sums[T]) for T in temperatures},
    "table_4_decimals": table_str,
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

print()
print("Resultados escritos en resultados.json")


Parte 1 - Softmax con temperatura: distribuciones y entropia (bits)
  T |     t0 |     t1 |     t2 |     t3 |     t4 |     t5 | entropia_bits
----+--------+--------+--------+--------+--------+--------+--------------
0.5 | 0.8231 | 0.1114 | 0.0410 | 0.0225 | 0.0020 | 0.0000 |        0.9147
  1 | 0.5516 | 0.2029 | 0.1231 | 0.0912 | 0.0275 | 0.0037 |        1.7998
  2 | 0.3583 | 0.2173 | 0.1693 | 0.1457 | 0.0800 | 0.0294 |        2.2888

Verificacion (suma de cada fila):
  T=0.5: suma = 1.0000
  T=1: suma = 1.0000
  T=2: suma = 1.0000

Valores (mismos numeros que en la tabla):
  T=0.5: t0=0.8231, t1=0.1114, t2=0.0410, t3=0.0225, t4=0.0020, t5=0.0000 | entropia=0.9147 bits
  T=1: t0=0.5516, t1=0.2029, t2=0.1231, t3=0.0912, t4=0.0275, t5=0.0037 | entropia=1.7998 bits
  T=2: t0=0.3583, t1=0.2173, t2=0.1693, t3=0.1457, t4=0.0800, t5=0.0294 | entropia=2.2888 bits

Resultados escritos en resultados.json


## Parte 2 — Muestreo de núcleo (top-p)

Sobre la distribución de T = 1 de la Parte 1 aplicamos top-p con p = 0.9: ordenamos los tokens por probabilidad descendente, conservamos el conjunto mínimo cuya probabilidad acumulada alcanza o supera 0.9 y renormalizamos sus probabilidades para que sumen 1. En la salida se ven las probabilidades ordenadas con sus acumuladas (que determinan dónde cae el corte), qué tokens sobreviven y cuáles quedan excluidos, y la distribución renormalizada con cuatro decimales.

In [2]:
# -*- coding: utf-8 -*-
"""
Subtask s2 (Parte 2) — Muestreo de núcleo (top-p) sobre la distribución de T = 1.

Recomputa de forma independiente (sin depender de archivos de la Parte 1):
  1) la softmax con temperatura numéricamente estable en T = 1 a partir de los
     logits z = [2.0, 1.0, 0.5, 0.2, -1.0, -3.0] para los tokens t0..t5,
  2) el corte top-p con p = 0.9: ordena los tokens por probabilidad descendente,
     conserva el conjunto mínimo cuya probabilidad acumulada alcanza o supera 0.9,
  3) la renormalización de las probabilidades de los tokens sobrevivientes.

Escribe todos los números en resultados.json y los imprime en stdout.
No se requiere figura para este subtask.
"""

import json
import numpy as np

# ----------------------------------------------------------------------
# Datos de la statement (Tarea B), embebidos literalmente
# ----------------------------------------------------------------------
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0], dtype=float)
T = 1.0          # temperatura de la Parte 1
p = 0.9          # umbral top-p de la Parte 2

# ----------------------------------------------------------------------
# Softmax con temperatura, numéricamente estable (misma implementación
# que la Parte 1: se resta el máximo de los logits antes de exponenciar)
# ----------------------------------------------------------------------
def softmax_with_temperature(z, temperature):
    z = np.asarray(z, dtype=float)
    z_shift = (z - np.max(z)) / temperature  # shift de estabilidad numérica
    e = np.exp(z_shift)
    return e / np.sum(e)

probs = softmax_with_temperature(logits, T)

# ----------------------------------------------------------------------
# Top-p (nucleus): ordenar por probabilidad descendente y acumular
# ----------------------------------------------------------------------
order = np.argsort(-probs, kind="stable")          # índices de mayor a menor prob.
sorted_tokens = [tokens[i] for i in order]
sorted_probs = probs[order]
cumulative = np.cumsum(sorted_probs)

# Conjunto mínimo cuya probabilidad acumulada alcanza o supera p
n_keep = 0
cum_at_cut = 0.0
for k in range(len(sorted_probs)):
    n_keep += 1
    cum_at_cut = float(cumulative[k])
    if cum_at_cut >= p:
        break

cum_before_cut = float(cumulative[n_keep - 2]) if n_keep >= 2 else 0.0
cut_token = sorted_tokens[n_keep - 1]

surviving_tokens = sorted_tokens[:n_keep]
excluded_tokens = sorted_tokens[n_keep:]
kept_probs = probs[order[:n_keep]]
renorm = kept_probs / kept_probs.sum()             # renormalización a suma 1

# ----------------------------------------------------------------------
# Reporte (floats planos, 4 decimales donde se pide)
# ----------------------------------------------------------------------
def d4(x):
    return float(np.round(float(x), 4))

results = {
    "logits": {tok: float(z) for tok, z in zip(tokens, logits)},
    "temperature": float(T),
    "p_threshold": float(p),
    "softmax_T1_full": {tok: float(pr) for tok, pr in zip(tokens, probs)},
    "softmax_T1_4dec": {tok: d4(pr) for tok, pr in zip(tokens, probs)},
    "sorted_tokens_desc": sorted_tokens,
    "sorted_probs_desc": {tok: float(pr) for tok, pr in zip(sorted_tokens, sorted_probs)},
    "cumulative_probs": {tok: float(c) for tok, c in zip(sorted_tokens, cumulative)},
    "cumulative_probs_4dec": {tok: d4(c) for tok, c in zip(sorted_tokens, cumulative)},
    "n_tokens_kept": int(n_keep),
    "cut_reached_at_token": cut_token,
    "cumulative_at_cut": cum_at_cut,
    "cumulative_before_cut": cum_before_cut,
    "surviving_tokens": surviving_tokens,
    "excluded_tokens": excluded_tokens,
    "renormalized_distribution_full": {tok: float(pr) for tok, pr in zip(surviving_tokens, renorm)},
    "renormalized_distribution_4dec": {tok: d4(pr) for tok, pr in zip(surviving_tokens, renorm)},
    "renormalized_sum_full": float(renorm.sum()),
    "renormalized_sum_4dec": float(sum(d4(pr) for pr in renorm)),
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

# ----------------------------------------------------------------------
# Salida por stdout (mismos números)
# ----------------------------------------------------------------------
print("Logits:", results["logits"])
print()
print(f"Softmax con temperatura (numerically stable) en T = {T}:")
for tok in tokens:
    print(f"  {tok}: p = {results['softmax_T1_4dec'][tok]:.4f}   (full: {results['softmax_T1_full'][tok]:.16f})")
print()
print(f"Top-p con p = {p}: tokens ordenados por probabilidad descendente")
print("  token |    p_i |  acumulada")
for tok in sorted_tokens:
    print(f"  {tok:>5} | {results['sorted_probs_desc'][tok]:.4f} | {results['cumulative_probs_4dec'][tok]:.4f}")
print()
print(f"Conjunto mínimo con probabilidad acumulada >= {p}:")
print(f"  acumulada antes del corte ({sorted_tokens[n_keep-2] if n_keep >= 2 else '-'}): "
      f"{cum_before_cut:.4f}  (< {p})")
print(f"  acumulada en el corte   ({cut_token}): {cum_at_cut:.4f}  (>= {p})")
print(f"  -> tokens sobrevivientes: {surviving_tokens}")
print(f"  -> tokens excluidos:      {excluded_tokens}")
print()
print("Distribución renormalizada sobre los tokens sobrevivientes (4 decimales):")
for tok in surviving_tokens:
    print(f"  {tok}: {results['renormalized_distribution_4dec'][tok]:.4f}")
print(f"  suma = {results['renormalized_sum_4dec']:.4f}")
print()
print("Resultados escritos en resultados.json")


Logits: {'t0': 2.0, 't1': 1.0, 't2': 0.5, 't3': 0.2, 't4': -1.0, 't5': -3.0}

Softmax con temperatura (numerically stable) en T = 1.0:
  t0: p = 0.5516   (full: 0.5516226378718239)
  t1: p = 0.2029   (full: 0.2029306277578035)
  t2: p = 0.1231   (full: 0.1230836475298394)
  t3: p = 0.0912   (full: 0.0911826087580713)
  t4: p = 0.0275   (full: 0.0274636739849860)
  t5: p = 0.0037   (full: 0.0037168040974761)

Top-p con p = 0.9: tokens ordenados por probabilidad descendente
  token |    p_i |  acumulada
     t0 | 0.5516 | 0.5516
     t1 | 0.2029 | 0.7546
     t2 | 0.1231 | 0.8776
     t3 | 0.0912 | 0.9688
     t4 | 0.0275 | 0.9963
     t5 | 0.0037 | 1.0000

Conjunto mínimo con probabilidad acumulada >= 0.9:
  acumulada antes del corte (t2): 0.8776  (< 0.9)
  acumulada en el corte   (t3): 0.9688  (>= 0.9)
  -> tokens sobrevivientes: ['t0', 't1', 't2', 't3']
  -> tokens excluidos:      ['t4', 't5']

Distribución renormalizada sobre los tokens sobrevivientes (4 decimales):
  t0: 0.5694
  t1

## Parte 3 — Comprobación empírica

Tomamos 10 000 muestras de la distribución renormalizada de la Parte 2 con numpy.random.default_rng(0). El gráfico de barras agrupadas compara, para cada token sobreviviente, la frecuencia observada con la probabilidad teórica (renormalizada), con etiquetas en los ejes, leyenda y nombres de los tokens en el eje x. Además calculamos la divergencia KL, en bits, de la distribución empírica respecto de la teórica usando solo los tokens sobrevivientes. Como muestra la salida, las barras son casi indistinguibles y la KL resulta prácticamente nula: el muestreo empírico reproduce la distribución teórica.

In [3]:
import json
import numpy as np
import matplotlib.pyplot as plt


def softmax_with_temperature(z, temperature):
    """Softmax con temperatura, numéricamente estable (resta el máximo)."""
    z = np.asarray(z, dtype=float)
    z_scaled = z / temperature
    z_shifted = z_scaled - np.max(z_scaled)
    e = np.exp(z_shifted)
    return e / np.sum(e)


# ----- Datos y parámetros (según el enunciado) -----
tokens = ["t0", "t1", "t2", "t3", "t4", "t5"]
logits = np.array([2.0, 1.0, 0.5, 0.2, -1.0, -3.0])
T = 1.0
p_threshold = 0.9
n_samples = 10000
seed = 0

# ----- Parte 1 (recomputada): softmax estable en T = 1 -----
probs_T1 = softmax_with_temperature(logits, T)

# ----- Parte 2 (recomputada): top-p con p = 0.9 -----
order_desc = np.argsort(-probs_T1, kind="stable")
sorted_tokens = [tokens[i] for i in order_desc]
sorted_probs = probs_T1[order_desc]
cumulative = np.cumsum(sorted_probs)

# conjunto mínimo cuya probabilidad acumulada alcanza p
n_kept = 0
cum = 0.0
for prob in sorted_probs:
    n_kept += 1
    cum += float(prob)
    if cum >= p_threshold:
        break

surviving_idx = order_desc[:n_kept]
surviving_tokens = [tokens[i] for i in surviving_idx]
surviving_probs = probs_T1[surviving_idx]
renorm = surviving_probs / np.sum(surviving_probs)

# ----- Parte 3: 10 000 muestras con numpy.random.default_rng(0) -----
rng = np.random.default_rng(seed)
sample_idx = rng.choice(len(surviving_tokens), size=n_samples, p=renorm)
counts = np.bincount(sample_idx, minlength=len(surviving_tokens))
empirical = counts / float(n_samples)

# ----- Divergencia KL en bits: D_KL(empírica || teórica), solo tokens supervivientes -----
mask = empirical > 0
kl_bits = float(np.sum(empirical[mask] * np.log2(empirical[mask] / renorm[mask])))
kl_bits_4dec = round(kl_bits, 4)

# ----- Gráfico de barras agrupadas: observado vs teórico -----
x = np.arange(len(surviving_tokens))
width = 0.38
fig, ax = plt.subplots(figsize=(8, 5.5))
b1 = ax.bar(x - width / 2, empirical, width,
            label="Frecuencia observada (10 000 muestras)",
            color="#4C72B0", edgecolor="black")
b2 = ax.bar(x + width / 2, renorm, width,
            label="Probabilidad teórica (top-p renormalizada)",
            color="#DD8452", edgecolor="black")
for bars in (b1, b2):
    for rect in bars:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width() / 2, h + 0.008, f"{h:.4f}",
                ha="center", va="bottom", fontsize=8)
ax.set_xlabel("Token")
ax.set_ylabel("Probabilidad / Frecuencia relativa")
ax.set_title("Parte 3 — Top-p (T=1, p=0.9): frecuencias observadas vs probabilidades teóricas\n"
             f"D_KL(empírica || teórica) = {kl_bits_4dec:.4f} bits")
ax.set_xticks(x)
ax.set_xticklabels(surviving_tokens)
ax.set_ylim(0, max(float(np.max(empirical)), float(np.max(renorm))) * 1.18)
ax.legend(loc="upper right")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("parte3_top_p_empirico.png", dpi=120)
plt.show()

# ----- Resultados -----
results = {
    "logits": {tokens[i]: float(logits[i]) for i in range(len(tokens))},
    "temperature": float(T),
    "p_threshold": float(p_threshold),
    "n_samples": int(n_samples),
    "rng_seed": int(seed),
    "softmax_T1_full": {tokens[i]: float(probs_T1[i]) for i in range(len(tokens))},
    "sorted_tokens_desc": sorted_tokens,
    "sorted_probs_desc": {sorted_tokens[i]: float(sorted_probs[i]) for i in range(len(sorted_tokens))},
    "cumulative_probs_desc": {sorted_tokens[i]: float(cumulative[i]) for i in range(len(sorted_tokens))},
    "n_tokens_kept": int(n_kept),
    "cut_reached_at_token": sorted_tokens[n_kept - 1],
    "cumulative_at_cut": float(cumulative[n_kept - 1]),
    "surviving_tokens": surviving_tokens,
    "renormalized_distribution_full": {surviving_tokens[i]: float(renorm[i]) for i in range(len(surviving_tokens))},
    "renormalized_distribution_4dec": {surviving_tokens[i]: round(float(renorm[i]), 4) for i in range(len(surviving_tokens))},
    "empirical_counts": {surviving_tokens[i]: int(counts[i]) for i in range(len(surviving_tokens))},
    "empirical_frequencies_full": {surviving_tokens[i]: float(empirical[i]) for i in range(len(surviving_tokens))},
    "empirical_frequencies_4dec": {surviving_tokens[i]: round(float(empirical[i]), 4) for i in range(len(surviving_tokens))},
    "kl_divergence_bits": kl_bits,
    "kl_divergence_bits_4dec": kl_bits_4dec,
}

with open("resultados.json", "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps(results, indent=2))


{
  "logits": {
    "t0": 2.0,
    "t1": 1.0,
    "t2": 0.5,
    "t3": 0.2,
    "t4": -1.0,
    "t5": -3.0
  },
  "temperature": 1.0,
  "p_threshold": 0.9,
  "n_samples": 10000,
  "rng_seed": 0,
  "softmax_T1_full": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "sorted_tokens_desc": [
    "t0",
    "t1",
    "t2",
    "t3",
    "t4",
    "t5"
  ],
  "sorted_probs_desc": {
    "t0": 0.5516226378718239,
    "t1": 0.2029306277578035,
    "t2": 0.12308364752983937,
    "t3": 0.09118260875807133,
    "t4": 0.02746367398498595,
    "t5": 0.003716804097476065
  },
  "cumulative_probs_desc": {
    "t0": 0.5516226378718239,
    "t1": 0.7545532656296273,
    "t2": 0.8776369131594667,
    "t3": 0.968819521917538,
    "t4": 0.9962831959025239,
    "t5": 1.0
  },
  "n_tokens_kept": 4,
  "cut_reached_at_token": "t3",
  "cumulative_at_cut": 0.9688195219175

## Parte 4 — Pregunta conceptual

**¿Qué le pasa a la distribución cuando T → 0 y cuando T → ∞, y por qué T → 0 equivale a la decodificación voraz (greedy)?**

La temperatura reescala los logits antes del softmax (p_i ∝ exp(z_i/T)): dividir entre T estira o comprime las diferencias entre logits, y con ello cambia la forma de la distribución.

**Cuando T → 0.** Las diferencias entre logits se amplifican al dividirlas entre una T diminuta, de modo que exp(z_max/T) domina absolutamente sobre todas las demás exponenciales. Toda la masa de probabilidad se concentra en el token de logit máximo (t0 en nuestro caso) y la distribución tiende a un delta: probabilidad 1 para el argmax y 0 para el resto. La tabla de la Parte 1 ya muestra la tendencia: al bajar la temperatura de T = 2 a T = 0.5, la probabilidad de t0 sube de 0.3583 a 0.8231, mientras que la de t5 cae de 0.0294 a 0.0000.

**Por qué esto equivale a greedy.** Para cualquier T, el token más probable de la distribución softmax coincide con el token de mayor logit (el softmax es monótono respecto de los logits). Con T > 0, sin embargo, el muestreo todavía puede desviarse del argmax; cuando T → 0 la probabilidad de los tokens no-argmax se anula, el muestreo se vuelve determinístico y siempre devuelve el argmax. Eso es exactamente la decodificación voraz: elegir en cada paso el token con mayor logit, sin aleatoriedad. Por eso T → 0 y greedy son equivalentes.

**Cuando T → ∞.** Dividir entre una T enorme aplana los logits (z_i/T → 0 para todos los tokens), todas las exponenciales valen prácticamente lo mismo y la distribución tiende a la uniforme sobre los seis tokens: todos se vuelven igual de probables sin importar su logit, y el muestreo se convierte en ruido puro.

**Evidencia: las entropías de la Parte 1.** La entropía mide la incertidumbre de la distribución y, como muestra la tabla de la Parte 1, crece monótonamente con la temperatura: 0.9147 bits con T = 0.5, 1.7998 bits con T = 1 y 2.2888 bits con T = 2. Al bajar T la entropía cae hacia 0 bits (incertidumbre nula, elección determinística, greedy); al subir T se acerca al máximo posible para un vocabulario de seis tokens, el de la distribución uniforme, log₂(6) bits. La tendencia de las entropías confirma ambos límites: T pequeña concentra la distribución y T grande la aplana.